In [1]:
!pip install stable-baselines3
!pip install aquacrop
!git clone https://github.com/aquacropos/aquacrop-gym.git
%cd aquacrop-gym
!pip install tensorflow




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 80.6 MB/s eta 0:00:00
Cloning into 'aquacrop-gym'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 129 (delta 26), reused 117 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (129/129), 18.46 MiB | 16.48 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/aquacrop-gym


In [2]:
import os
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from gymnasium import Env
from gymnasium.spaces import Discrete, Box
import numpy as np
import random
import aquacropgym

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:
import aquacrop

from aquacrop import AquaCropModel, Soil, Crop, InitialWaterContent
from aquacrop.utils import prepare_weather, get_filepath
weather_file_path = get_filepath('tunis_climate.txt')


In [123]:
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces

from aquacrop import AquaCropModel, Soil, Crop, InitialWaterContent, IrrigationManagement
from aquacrop.utils import prepare_weather, get_filepath


class AquaCropIrrigationEnv(gym.Env):
    def __init__(self):
        super().__init__()
        self.max_season_water = 4000.0  # лимит воды за сезон в мм
        self.weather = prepare_weather(get_filepath("tunis_climate.txt"))
        self.weather["Precipitation"] = 0.0

        self.start_date = "1979-04-01"
        self.end_date = "1979-07-20"

        self.dates = pd.date_range(self.start_date, self.end_date, freq="D")
        self.max_days = len(self.dates)

        self.action_space = spaces.Box(
            low=np.array([0.0], dtype=np.float32),
            high=np.array([1.0], dtype=np.float32),
            dtype=np.float32
        )

        self.observation_space = spaces.Box(
          low=np.zeros(8, dtype=np.float32),
          high=np.ones(8, dtype=np.float32),
          dtype=np.float32
        )

        self.day = 0
        self.total_water = 0.0
        self.irrigation_schedule = None
        self.model = None

    def _make_schedule(self):
      return pd.DataFrame({
          "Date": pd.date_range(self.start_date, self.end_date, freq="D"),
          "Depth": np.zeros(self.max_days, dtype=float)
      })

    def _build_model(self):
      irrigation = IrrigationManagement(
          irrigation_method=5,
          depth=0.0,
          MaxIrr=30.0
      )

      model = AquaCropModel(
          sim_start_time=self.start_date.replace("-", "/"),
          sim_end_time=self.end_date.replace("-", "/"),
          weather_df=self.weather,
          soil=Soil("SandyLoam"),
          crop=Crop("Tomato", planting_date="04/01"),
          initial_water_content=InitialWaterContent(value=["WP"]),
          irrigation_management=irrigation
      )

      return model

    def reset(self, seed=None, options=None):
      super().reset(seed=seed)

      self.day = 0
      self.total_water = 0
      self.total_yield = 0
      self.total_yield = 0.0
      self.model = self._build_model()

      # Важно: просто инициализируем модель, не пересоздаём её потом каждый день
      self.model._initialize()

      return self._get_obs(), {}

    def step(self, action):
      # Нельзя делать следующий шаг после завершения AquaCrop
      if self.model._clock_struct.model_is_finished:
          raise RuntimeError(
              "AquaCrop model is already finished. Call env.reset() "
              "before starting a new episode."
          )

      raw_action = float(np.asarray(action).reshape(-1)[0])
      raw_action = float(np.clip(raw_action, 0.0, 1.0))

      requested_irrigation = raw_action * 30.0

      remaining_water = max(
          self.max_season_water - self.total_water,
          0.0
      )

      irrigation = min(
          requested_irrigation,
          remaining_water
      )

      prev_biomass = float(self.model._init_cond.biomass)
      prev_irr_cum = float(self.model._init_cond.irr_cum)
      prev_th = float(self.model._init_cond.th[0])

      # Устанавливаем полив на текущий день
      self.model._param_struct.IrrMngt.depth = irrigation

      # По умолчанию num_steps=1, но лучше написать явно
      self.model.run_model(
          num_steps=1,
          initialize_model=False
      )

      biomass_now = float(self.model._init_cond.biomass)
      irr_cum_now = float(self.model._init_cond.irr_cum)
      th1 = float(self.model._init_cond.th[0])

      biomass_gain = biomass_now - prev_biomass
      actual_irrigation_today = max(
          irr_cum_now - prev_irr_cum,
          0.0
      )

      self.total_water += actual_irrigation_today

      reward = 0.0

      # Награда за рост биомассы
      reward += biomass_gain

      # До полива почва была сухой, и агент полил
      if prev_th < 0.23 and irrigation >= 5.0:
          reward += 5.0

      # Хорошая влажность после действия
      if 0.25 <= th1 <= 0.36:
          reward += 8.0

      # Слишком сухая почва
      if th1 < 0.22:
          reward -= 20.0

      # Слишком влажная почва
      if th1 > 0.40:
          reward -= 5.0

      # Штрафовать нужно фактически применённую воду,
      # а не только запрошенную
      reward -= actual_irrigation_today*1.2

      self.day += 1

      # Проверяем завершение именно внутри AquaCrop
      model_finished = self.model.get_additional_information()[
          "has_model_finished"
      ]

      calendar_finished = self.day >= self.max_days
      water_finished = self.total_water >= self.max_season_water

      terminated = bool(
          model_finished
          or calendar_finished
          or water_finished
      )

      truncated = False
      info = {
          "day": self.day,
          "irrigation_today": actual_irrigation_today,
          "total_water": self.total_water,
          "soil_moisture": th1,
          "biomass": biomass_now,
          "model_finished": model_finished,
      }

      if terminated:
          # Не вызываем till_termination=True, если AquaCrop
          # уже сам завершил сезон
          if not model_finished:
              self.model.run_model(
                  till_termination=True,
                  initialize_model=False
              )

          results = self.model.get_simulation_results()

          if (
              results is not False
              and results is not None
              and len(results) > 0
          ):
              self.final_yield = float(
                  results["Dry yield (tonne/ha)"].iloc[-1]
              )
          else:
              self.final_yield = 0.0

          self.total_yield = self.final_yield

          reward += self.final_yield * 30.0

          if self.final_yield <= 0:
              reward -= 7000.0

          info["final_yield"] = self.final_yield
          info["termination_reason"] = (
              "crop_finished"
              if model_finished
              else "water_limit"
              if water_finished
              else "calendar_finished"
          )

      # После увеличения self.day индекс может стать равным max_days.
      # Поэтому на терминальном шаге используем последний допустимый день.
      obs = self._get_obs()

      return obs, float(reward), terminated, truncated, info

    def _get_obs(self):
        th1 = float(self.model._init_cond.th[0])
        canopy = float(self.model._init_cond.canopy_cover)
        biomass = float(self.model._init_cond.biomass)

        th1_norm = np.clip(th1 / 0.45, 0.0, 1.0)
        canopy_norm = np.clip(canopy, 0.0, 1.0)
        biomass_norm = np.clip(biomass / 20000.0, 0.0, 1.0)

        weather_row = self.weather.iloc[self.day]

        tmin = float(weather_row["MinTemp"])
        tmax = float(weather_row["MaxTemp"])
        ref_et = float(weather_row["ReferenceET"])

        avg_temp = (tmin + tmax) / 2.0

        # temperature normalization
        temp_norm = np.clip((avg_temp + 5.0) / 50.0, 0.0, 1.0)

        # light proxy: AquaCrop weather does not directly give sunlight,
        # so ReferenceET is used as a proxy for heat/light demand
        light_norm = np.clip(ref_et / 8.0, 0.0, 1.0)

        # humidity proxy: high ET usually means drier air
        humidity_norm = np.clip(1.0 - (ref_et / 8.0), 0.0, 1.0)

        day_norm = np.clip(self.day / self.max_days, 0.0, 1.0)
        # soil temperature proxy
        soil_temp = avg_temp - 2.0
        soil_temp_norm = np.clip((soil_temp + 5.0) / 50.0, 0.0, 1.0)
        return np.array([day_norm,th1_norm, canopy_norm, biomass_norm, temp_norm,light_norm, humidity_norm,soil_temp_norm,], dtype=np.float32)

    def render(self):
        print(f"Day: {self.day}, Total water: {self.total_water}")

In [103]:
env = AquaCropIrrigationEnv()

obs, info = env.reset()

total_reward = 0.0

for i in range(env.max_days):
    action = np.array([0.25], dtype=np.float32)

    obs, reward, terminated, truncated, info = env.step(action)

    total_reward += reward

    print("day:", i)
    print("obs:", obs)
    print("reward:", reward)
    print("irr_cum:", env.total_water)
    print("th1:", env.model._init_cond.th[0])
    print()

    if terminated or truncated:
        print("Episode finished")
        print("Reason:", info.get("termination_reason"))
        print("Total water:", info.get("total_water"))
        print("Final yield:", info.get("final_yield"))
        print("Total reward:", total_reward)
        break

day: 0
obs: [0.00900901 0.32777777 0.         0.         0.33       0.1625
 0.8375     0.29      ]
reward: -21.0
irr_cum: 7.5
th1: 0.14750000000000005

day: 1
obs: [0.01801802 0.4308889  0.         0.         0.25       0.15
 0.85       0.21      ]
reward: -21.0
irr_cum: 15.0
th1: 0.19390000000000004

day: 2
obs: [0.02702703 0.52177775 0.         0.         0.34       0.225
 0.775      0.3       ]
reward: -1.0
irr_cum: 22.5
th1: 0.23480000000000004

day: 3
obs: [3.60360369e-02 5.70359051e-01 7.53809884e-03 1.17036125e-05
 3.70000005e-01 1.74999997e-01 8.24999988e-01 3.30000013e-01]
reward: 2.2340722408341787
irr_cum: 30.0
th1: 0.25666156614196234

day: 4
obs: [4.5045044e-02 5.6878865e-01 8.5235247e-03 2.4929614e-05 4.0000001e-01
 2.2499999e-01 7.7499998e-01 3.6000001e-01]
reward: 2.2645200255621134
irr_cum: 37.5
th1: 0.2559548865467034

day: 5
obs: [5.4054055e-02 5.8336353e-01 9.6377721e-03 3.9874914e-05 3.4000000e-01
 8.7499999e-02 9.1250002e-01 3.0000001e-01]
reward: 2.29890603526648

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


day: 52
obs: [0.4774775  0.5230641  0.70430493 0.01590551 0.33       0.225
 0.775      0.29      ]
reward: 8.871895425842297
irr_cum: 397.5
th1: 0.235378826408361

day: 53
obs: [0.4864865  0.51800096 0.7095383  0.01665269 0.27       0.1625
 0.8375     0.23      ]
reward: 8.943628549769983
irr_cum: 405.0
th1: 0.2331004351860116

day: 54
obs: [0.4954955  0.51764095 0.7141665  0.01740303 0.29       0.1375
 0.8625     0.25      ]
reward: 9.006768330572527
irr_cum: 412.5
th1: 0.23293843397848848

day: 55
obs: [0.5045045  0.51806194 0.71825963 0.01815615 0.32       0.225
 0.775      0.28      ]
reward: 9.062376074657038
irr_cum: 420.0
th1: 0.2331278613073883

day: 56
obs: [0.5135135  0.5184919  0.7218794  0.01891172 0.26       0.1375
 0.8625     0.22      ]
reward: 9.111374742706573
irr_cum: 427.5
th1: 0.23332137781022647

day: 57
obs: [0.5225225  0.52008015 0.7250806  0.01966945 0.28       0.225
 0.775      0.24      ]
reward: 9.15456860523824
irr_cum: 435.0
th1: 0.23403605663056806

day: 5

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [124]:
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
env = DummyVecEnv([
    lambda: Monitor(AquaCropIrrigationEnv())
])

# модель будет видеть последние 4 состояния
env = VecFrameStack(env, n_stack=5)

policy_kwargs = dict(
    net_arch=dict(
        pi=[128, 128],
        vf=[128, 128]
    )
)

agent = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    n_steps=128,
    batch_size=64,
    learning_rate=0.0005,
    ent_coef=0.01)

agent.learn(total_timesteps=10000)

Using cpu device


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 110      |
|    ep_rew_mean     | -871     |
| time/              |          |
|    fps             | 243      |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 128      |
---------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 110          |
|    ep_rew_mean          | -951         |
| time/                   |              |
|    fps                  | 232          |
|    iterations           | 2            |
|    time_elapsed         | 1            |
|    total_timesteps      | 256          |
| train/                  |              |
|    approx_kl            | 0.0035765632 |
|    clip_fraction        | 0.00234      |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.42        |
|    explained_variance   | -0.000829    |
|    learning_rate        | 0.0005       |
|    loss                 | 1.01e+04     |
|    n_updates            | 10           |
|    policy_gradient_loss | -0.00709     |
|    std                  | 1.01         |
|    value_loss           | 2.05e+04     |
------------------------------------------


...

--------------------------------------------
| rollout/                |                |
|    ep_len_mean          | 109            |
|    ep_rew_mean          | -1.09e+03      |
| time/                   |                |
|    fps                  | 190            |
|    iterations           | 77             |
|    time_elapsed         | 51             |
|    total_timesteps      | 9856           |
| train/                  |                |
|    approx_kl            | 0.000107264146 |
|    clip_fraction        | 0              |
|    clip_range           | 0.2            |
|    entropy_loss         | -1.47          |
|    explained_variance   | -1.44e-05      |
|    learning_rate        | 0.0005         |
|    loss                 | 8.97e+05       |
|    n_updates            | 760            |
|    policy_gradient_loss | 0.000758       |
|    std                  | 1.05           |
|    value_loss           | 1.55e+06       |
--------------------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 109           |
|    ep_rew_mean          | -1.09e+03     |
| time/                   |               |
|    fps                  | 190           |
|    iterations           | 78            |
|    time_elapsed         | 52            |
|    total_timesteps      | 9984          |
| train/                  |               |
|    approx_kl            | 2.3667235e-05 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -1.47         |
|    explained_variance   | -5.71e-05     |
|    learning_rate        | 0.0005        |
|    loss                 | 1.12e+04      |
|    n_updates            | 770           |
|    policy_gradient_loss | -0.000246     |
|    std                  | 1.05          |
|    value_loss           | 1.75e+04      |
-------------------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 109          |
|    ep_rew_mean          | -1.09e+03    |
| time/                   |              |
|    fps                  | 190          |
|    iterations           | 79           |
|    time_elapsed         | 53           |
|    total_timesteps      | 10112        |
| train/                  |              |
|    approx_kl            | 0.0012645451 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.47        |
|    explained_variance   | -5.81e-05    |
|    learning_rate        | 0.0005       |
|    loss                 | 9.51e+03     |
|    n_updates            | 780          |
|    policy_gradient_loss | -0.00175     |
|    std                  | 1.05         |
|    value_loss           | 1.98e+04     |
------------------------------------------


In [125]:
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack

env = DummyVecEnv([
    lambda: Monitor(AquaCropIrrigationEnv())
])

env = VecFrameStack(env, n_stack=5)

obs = env.reset()
done = False

total_reward = 0.0
actions = []

final_water = None
final_yield = None

while not done:
    action, _ = agent.predict(obs, deterministic=True)

    actions.append(float(action[0][0]))

    obs, reward, dones, infos = env.step(action)

    total_reward += float(reward[0])
    done = bool(dones[0])

    if done:
        print("Final info:", infos[0])
        final_water = infos[0].get("total_water")
        final_yield = infos[0].get("final_yield")

print("Total reward:", total_reward)
print("Total water:", final_water)
print("Total yield:", final_yield)
print("Min action:", min(actions))
print("Max action:", max(actions))
print("Avg action:", sum(actions) / len(actions))
print(actions)

Final info: {'day': 110, 'irrigation_today': 11.29288136959076, 'total_water': 737.9526217281818, 'soil_moisture': 0.26384989939408254, 'biomass': 760.4422445194464, 'model_finished': True, 'final_yield': 3.688847480694854, 'termination_reason': 'crop_finished', 'episode': {'r': -713.435477, 'l': 110, 't': 0.327642}, 'TimeLimit.truncated': False, 'terminal_observation': array([0.954955  , 0.60157883, 0.74762475, 0.03496891, 0.39      ,
       0.3875    , 0.6125    , 0.35      , 0.963964  , 0.59079516,
       0.74789935, 0.03573318, 0.35      , 0.325     , 0.675     ,
       0.31      , 0.972973  , 0.5825181 , 0.74814224, 0.03649681,
       0.35      , 0.3625    , 0.6375    , 0.31      , 0.981982  ,
       0.6145601 , 0.748357  , 0.0372598 , 0.35      , 0.3625    ,
       0.6375    , 0.31      , 0.990991  , 0.5863331 , 0.74854696,
       0.03802211, 0.37      , 0.3875    , 0.6125    , 0.33      ],
      dtype=float32)}
Total reward: -713.4354669451714
Total water: 737.9526217281818
Tota

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [122]:
agent.save("ppo_aquacrop_irrigation_model(Tomato_W_674)")

In [ ]:
from google.colab import files
from stable_baselines3 import PPO

env = AquaCropIrrigationEnv()
agent = PPO.load("ppo_aquacrop_irrigation_model(W_234)", env=env)
obs, info = env.reset()
done = False

total_reward = 0
actions = []

while not done:
    action, _ = agent.predict(obs, deterministic=True)
    actions.append(float(action[0]))

    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward

    done = terminated or truncated

print("Total reward:", total_reward)
print("Total water:", env.total_water)
print("Total yield:", env.final_yield)
print("Min action:", min(actions))
print("Max action:", max(actions))
print("Avg action:", sum(actions) / len(actions))
print(actions)

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


ValueError: Observation spaces do not match: Box(0.0, 1.0, (40,), float32) != Box(0.0, 1.0, (8,), float32)